# ⚠ DEPRECATED EXPLORATORY NOTEBOOK — DO NOT USE FOR THE REPORTED RESULTS

**This notebook is retained only to document how the study's design evolved.**

It is **not** the experiment behind any number in the manuscript. In particular, its
`P3` encryption formulation is **methodologically invalid**: expressing "no encryption"
through a cross-file `count` leaves the encryption block present in the static source,
so it is not a clean unencrypted-bucket test. That property is excluded from the
reported results, and the manifest records the exclusion.

Earlier versions here also compared only rule-identifier sets, without matching the
expected resource address, which produced verdicts later shown to be wrong.

---

**Use `notebooks/RQ4_Canonical_Experiment.ipynb` instead.** It is the single declared
run path: four-part oracle with resource-address matching, distinct
RESOLVED / NOT_RESOLVED / INCONCLUSIVE / ERROR / N/A verdicts, full capture of every
tool execution, an emitted environment manifest, and HCL validation before scanning.


# Exploratory round — Multi-tool RQ4 + Dataset Distributions + Unresolved Breakdown

Run all. Three independent parts:

1. **Phase 6e** — repeats the 7-construct cross-file experiment on **Checkov + tfsec + KICS**
   (answers "why Checkov only?"). Installs the three tools first.
2. **Phase 7A** — TerraDS distributions (providers, size, stars, age, modules).
3. **Phase 7B** — categorised breakdown of the 4.5% unresolved cross-file edges.

Parts 7A/7B need `TerraDS.sqlite` (run the Explorer first). Part 6e is self-contained.

In [ ]:
# --- install the three scanners (2-4 min) ---
!pip -q install checkov
# tfsec
!curl -s -L https://github.com/aquasecurity/tfsec/releases/latest/download/tfsec-linux-amd64 -o /usr/local/bin/tfsec && chmod +x /usr/local/bin/tfsec
# kics (binary + assets)
!curl -s -L https://github.com/Checkmarx/kics/releases/latest/download/kics_linux_x64.tar.gz -o /tmp/kics.tar.gz && mkdir -p /opt/kics && tar -xzf /tmp/kics.tar.gz -C /opt/kics && ln -sf /opt/kics/kics /usr/local/bin/kics
import os; os.environ["KICS_QUERIES_PATH"]="/opt/kics/assets/queries"
print("tools installed")
!tfsec --version; kics version 2>/dev/null | head -1; checkov --version

## Phase 6e — Checkov + tfsec + KICS on the 7 constructs

In [ ]:
"""
Phase 6e — Multi-tool cross-file resolution (Checkov + tfsec + KICS).

Extends Phase 6d from one scanner to three, addressing the strongest generalisability
question: "why Checkov only?". Same seven constructs, same control/treatment/inline
design, same decision rule — now applied to each tool. The output is a
construct x tool matrix of RESOLVED / NOT RESOLVED / INCONCLUSIVE verdicts.

Tools:
  - checkov  (json)    : pip install checkov
  - tfsec    (json)    : binary from aquasecurity (installed below)
  - kics     (json)    : binary from Checkmarx    (installed below)

Each tool has its own "signal check": the rule id that fires for a public-read S3
ACL. We DISCOVER that signal per tool from the inline positive control rather than
hard-coding it, so the experiment is robust to differing rule identifiers.
"""
import os, subprocess, json, shutil, glob, re

WORK = "/content/multitool" if os.path.isdir("/content") else "./multitool"
os.makedirs(WORK, exist_ok=True)
SECURE, INSECURE = "private", "public-read"

def sh(cmd, timeout=1200, **kw):
    return subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, **kw)

# ----------------------------------------------------------------- construct builders
def w(path, rel, text):
    full=os.path.join(path,rel); os.makedirs(os.path.dirname(full),exist_ok=True)
    open(full,"w").write(text)
def fresh(name):
    d=os.path.join(WORK,name); shutil.rmtree(d,ignore_errors=True); os.makedirs(d); return d

BUCKET='resource "aws_s3_bucket" "data" {\n  bucket = "example-data-bucket"\n}\n\n'
ACL=lambda expr: ('resource "aws_s3_bucket_acl" "data" {\n  bucket = aws_s3_bucket.data.id\n'
                  f'  acl    = {expr}\n' '}\n')

def c_inline(n,v):
    d=fresh(n); w(d,"main.tf",BUCKET+ACL(f'"{v}"')); return d
def c_var_default(n,v):
    d=fresh(n); w(d,"variables.tf",f'variable "bucket_acl" {{\n  type=string\n  default="{v}"\n}}\n')
    w(d,"main.tf",BUCKET+ACL("var.bucket_acl")); return d
def c_locals(n,v):
    d=fresh(n); w(d,"locals.tf",f'locals {{\n  effective_acl="{v}"\n}}\n')
    w(d,"main.tf",BUCKET+ACL("local.effective_acl")); return d
def c_tfvars(n,v):
    d=fresh(n); w(d,"variables.tf",f'variable "bucket_acl" {{\n  type=string\n  default="{SECURE}"\n}}\n')
    w(d,"terraform.tfvars",f'bucket_acl="{v}"\n'); w(d,"main.tf",BUCKET+ACL("var.bucket_acl")); return d
def c_module_input(n,v):
    d=fresh(n)
    w(d,"main.tf",'module "storage" {\n  source="./modules/storage"\n'+f'  bucket_acl="{v}"\n'+'}\n')
    w(d,"modules/storage/variables.tf",'variable "bucket_acl" {\n  type=string\n  default="private"\n}\n')
    w(d,"modules/storage/main.tf",BUCKET+ACL("var.bucket_acl")); return d
def c_module_chain(n,v):
    d=fresh(n)
    w(d,"main.tf",'module "cfg" {\n  source="./modules/cfg"\n'+f'  acl_in="{v}"\n'+'}\n\n'
      'module "storage" {\n  source="./modules/storage"\n  bucket_acl=module.cfg.acl_out\n}\n')
    w(d,"modules/cfg/variables.tf",'variable "acl_in" {\n  type=string\n}\n')
    w(d,"modules/cfg/outputs.tf",'output "acl_out" {\n  value=var.acl_in\n}\n')
    w(d,"modules/storage/variables.tf",'variable "bucket_acl" {\n  type=string\n  default="private"\n}\n')
    w(d,"modules/storage/main.tf",BUCKET+ACL("var.bucket_acl")); return d
def c_nested(n,v):
    d=fresh(n)
    w(d,"main.tf",'module "outer" {\n  source="./modules/outer"\n'+f'  bucket_acl="{v}"\n'+'}\n')
    w(d,"modules/outer/variables.tf",'variable "bucket_acl" {\n  type=string\n}\n')
    w(d,"modules/outer/main.tf",'module "inner" {\n  source="./inner"\n  bucket_acl=var.bucket_acl\n}\n')
    w(d,"modules/outer/inner/variables.tf",'variable "bucket_acl" {\n  type=string\n  default="private"\n}\n')
    w(d,"modules/outer/inner/main.tf",BUCKET+ACL("var.bucket_acl")); return d
def c_override(n,v):
    d=fresh(n); w(d,"main.tf",BUCKET+ACL(f'"{SECURE}"'))
    w(d,"override.tf",'resource "aws_s3_bucket_acl" "data" {\n'+f'  acl="{v}"\n'+'}\n'); return d

CONSTRUCTS=[("C1 variable default",c_var_default),
            ("C2 local value",c_locals),
            ("C3 terraform.tfvars",c_tfvars),
            ("C4 module input",c_module_input),
            ("C5 module output chaining",c_module_chain),
            ("C6 nested module (2 levels)",c_nested),
            ("C7 override.tf",c_override)]

# ----------------------------------------------------------------- per-tool runners
MODULE_PREFIX = re.compile(r"^(?:module\.[A-Za-z0-9_-]+\.)+")

def run_checkov(path):
    r=sh(["checkov","-d",path,"-o","json","--compact","--quiet"])
    out=r.stdout.strip()
    if not out: return set()
    try: res=json.loads(out)
    except Exception: return set()
    blocks=res if isinstance(res,list) else [res]
    ids=set()
    for b in blocks:
        for f in b.get("results",{}).get("failed_checks",[]):
            ids.add(f.get("check_id"))
    return ids

def run_tfsec(path):
    r=sh(["tfsec",path,"-f","json","--no-colour"])
    out=r.stdout.strip()
    if not out: return set()
    try: res=json.loads(out)
    except Exception: return set()
    ids=set()
    for res_item in (res.get("results") or []):
        # tfsec rule ids like aws-s3-no-public-access-with-acl
        ids.add(res_item.get("long_id") or res_item.get("rule_id"))
    return ids

def run_kics(path):
    outdir=os.path.join(path,"_kics"); os.makedirs(outdir,exist_ok=True)
    qp=os.environ.get("KICS_QUERIES_PATH")
    kics_cmd=["kics","scan","-p",path,"--report-formats","json","-o",outdir,"--silent","--no-progress"]
    if qp: kics_cmd+=["-q",qp]
    r=sh(kics_cmd, timeout=1800)
    js=glob.glob(os.path.join(outdir,"*.json"))
    if not js: return set()
    try: res=json.load(open(js[0]))
    except Exception: return set()
    ids=set()
    for q in res.get("queries",[]):
        # only count queries that actually have results
        if q.get("files"):
            ids.add(q.get("query_id") or q.get("query_name"))
    return ids

TOOLS={"checkov":run_checkov,"tfsec":run_tfsec,"kics":run_kics}

def verdict(signal, ctrl, treat):
    if not signal: return "INCONCLUSIVE (no signal)"
    if signal.issubset(treat) and not signal.issubset(ctrl): return "RESOLVED"
    if signal.isdisjoint(treat - ctrl): return "NOT RESOLVED"
    return "PARTIAL"

def which(tool): return shutil.which(tool) is not None

def main():
    print("="*80); print("PHASE 6e — cross-file resolution across Checkov, tfsec, KICS"); print("="*80)
    available={name:which(name) for name in TOOLS}
    print("tool availability:", available)

    # discover each tool's signal from the inline positive control
    isec=c_inline("inline_secure",SECURE)
    iins=c_inline("inline_insecure",INSECURE)
    signals={}
    for name,run in TOOLS.items():
        if not available[name]: continue
        s=run(iins)-run(isec)
        signals[name]=s
        print(f"  [{name}] signal check(s) for public-read ACL: {sorted(s) if s else 'NONE (rule missing?)'}")

    matrix={}
    for label,builder in CONSTRUCTS:
        tag=label.split()[0]
        matrix[label]={}
        for name,run in TOOLS.items():
            if not available[name] or not signals.get(name):
                matrix[label][name]="N/A"; continue
            ctrl=run(builder(f"{tag}_{name}_ctrl",SECURE))
            treat=run(builder(f"{tag}_{name}_treat",INSECURE))
            matrix[label][name]=verdict(signals[name],ctrl,treat)

    print("\n"+"="*80); print("RESULT MATRIX (construct x tool)"); print("="*80)
    tools=[t for t in TOOLS if available[t] and signals.get(t)]
    print(f"{'Construct':32s} " + " ".join(f"{t:>14s}" for t in tools))
    print("-"*80)
    for label in matrix:
        print(f"{label:32s} " + " ".join(f"{matrix[label][t]:>14s}" for t in tools))

    json.dump({"signals":{k:sorted(v) for k,v in signals.items()},"matrix":matrix},
              open(os.path.join(WORK,"multitool_results.json"),"w"),indent=2)
    print(f"\nSaved multitool_results.json to {WORK}")
    print("\nREAD: a construct RESOLVED by all available tools is strongly resolved;")
    print("one NOT RESOLVED across tools is a robust cross-tool blind spot.")

if __name__=="__main__":
    main()


## Phase 7 — Dataset distributions (A) + unresolved-edge breakdown (B)

In [ ]:
"""
Phase 7 — Dataset distributions + unresolved-edge breakdown (methodological validation).

Part A: characterise the TerraDS corpus so readers can judge generalisability:
  - cloud provider distribution (from Modules.Providers)
  - repository size, stars, forks distributions (quartiles)
  - repository age (CreatedAt -> years) and recency (LatestCommitAt)
  - modules-per-repo distribution
Part B: classify WHY 4.5% of local cross-file edges did not resolve to an in-repo
  module, turning an unexplained residual into a categorised breakdown:
  - parent_traversal_escapes_repo : '../' path climbs above the indexed module set
  - target_dir_not_indexed        : resolved path has no matching module row
  - malformed_or_empty_source     : blank/º unpardable source
  - non_local (safety check)      : should be zero here (locals only)
"""
import sqlite3, json, os, glob, re, statistics as st
from collections import Counter, defaultdict

def find_db():
    for p in ["data/terrads/TerraDS.sqlite","/content/terrads/TerraDS.sqlite","/tmp/mock/TerraDS.sqlite"]:
        if os.path.exists(p): return p
    h=glob.glob("/content/**/TerraDS.sqlite",recursive=True)+glob.glob("**/TerraDS.sqlite",recursive=True)
    return h[0] if h else None

def quart(xs):
    xs=sorted(x for x in xs if x is not None)
    if not xs: return None
    n=len(xs)
    q=lambda p: xs[min(n-1,int(p*n))]
    return {"min":xs[0],"q1":q(0.25),"median":q(0.5),"q3":q(0.75),
            "p90":q(0.90),"max":xs[-1],"mean":round(sum(xs)/n,1)}

# ---------------- Part A ----------------
def part_a(con):
    print("="*70); print("PART A — DATASET DISTRIBUTIONS"); print("="*70)

    # provider distribution
    prov=Counter()
    for (p,) in con.execute("SELECT Providers FROM Modules WHERE Providers IS NOT NULL AND Providers!='[]'"):
        try:
            for x in json.loads(p): prov[x]+=1
        except Exception: pass
    total_prov=sum(prov.values()) or 1
    print("\nCloud/provider distribution (top 12 by module count):")
    for name,c in prov.most_common(12):
        print(f"  {name:16s} {c:8d}  ({100*c/total_prov:.1f}%)")

    # repo-level metadata (defensive: some columns may be absent in variant schemas)
    cols=[c[1] for c in con.execute("PRAGMA table_info('Repositories')").fetchall()]
    want=[c for c in ["StarCount","ForkCount","SizeInKb","CreatedAt","LatestCommitAt","Archived"] if c in cols]
    rows=con.execute(f"SELECT {','.join(want)} FROM Repositories").fetchall()
    col_idx={name:i for i,name in enumerate(want)}
    def col(r,name): 
        i=col_idx.get(name); return r[i] if i is not None else None
    stars=[col(r,"StarCount") or 0 for r in rows]
    forks=[col(r,"ForkCount") or 0 for r in rows]
    size=[col(r,"SizeInKb") or 0 for r in rows]
    print("\nRepository size (KB):", quart(size))
    print("Stars:", quart(stars))
    print("Forks:", quart(forks))

    import datetime as dt
    def year(s):
        try: return int(str(s)[:4])
        except Exception: return None
    created=[year(col(r,"CreatedAt")) for r in rows]; created=[c for c in created if c]
    latest=[year(col(r,"LatestCommitAt")) for r in rows]; latest=[c for c in latest if c]
    if created:
        cc=Counter(created)
        print("\nRepository creation year (distribution):")
        for y in sorted(cc): print(f"  {y}: {cc[y]}")
    if latest:
        lc=Counter(latest)
        print("\nLatest commit year (recency):")
        for y in sorted(lc): print(f"  {y}: {lc[y]}")
    archived=sum(1 for r in rows if col(r,"Archived")==1)
    print(f"\nArchived repositories: {archived} ({100*archived/len(rows):.1f}%)")

    # modules per repo
    mpr=[c for (c,) in con.execute("SELECT COUNT(*) FROM Modules GROUP BY RepositoryId")]
    print("\nModules per repository:", quart(mpr))

# ---------------- Part B ----------------
def classify_source(src):
    s=src.strip()
    if s.startswith("./"): return "local_subdir"
    if s.startswith("../"): return "local_traversal"
    if s.startswith(("git::","github.com","git@")) or ".git" in s: return "vcs_remote"
    if s.startswith(("http://","https://")): return "http_remote"
    if s.startswith(("s3::","gcs::","oss::")): return "cloud_bucket"
    if re.match(r"^[A-Za-z0-9_.-]+/[A-Za-z0-9_./-]+$",s) and not s.startswith("."): return "registry"
    return "other"

def part_b(con):
    print("\n"+"="*70); print("PART B — WHY 4.5% OF CROSS-FILE EDGES DON'T RESOLVE"); print("="*70)
    mods=con.execute("SELECT Id,RepositoryId,Path,ModuleCalls FROM Modules").fetchall()
    by_repo=defaultdict(list)
    for m in mods: by_repo[m[1]].append(m)

    reasons=Counter(); total_cf=0; resolved=0; examples=defaultdict(list)
    for repo_id, modlist in by_repo.items():
        path_index={ (m[2] or "").strip("/").replace("\\","/"): m[0] for m in modlist }
        for m in modlist:
            raw=m[3]
            if not raw or raw in ("[]",""): continue
            try: calls=json.loads(raw)
            except Exception: continue
            sd=(m[2] or "").strip("/").replace("\\","/")
            for call in calls:
                src=(call.get("source") or "")
                kind=classify_source(src)
                if kind not in ("local_subdir","local_traversal"): continue
                total_cf+=1
                if not src.strip():
                    reasons["malformed_or_empty_source"]+=1; continue
                joined=os.path.normpath(os.path.join(sd,src)).replace("\\","/")
                tgt=(joined[2:] if joined.startswith("./") else joined).strip("/")
                if tgt in path_index:
                    resolved+=1; continue
                # unresolved: categorise
                if joined.startswith("..") or tgt.startswith(".."):
                    reasons["parent_traversal_escapes_repo"]+=1
                    if len(examples["parent_traversal_escapes_repo"])<5:
                        examples["parent_traversal_escapes_repo"].append(f"{sd} + {src}")
                else:
                    reasons["target_dir_not_indexed"]+=1
                    if len(examples["target_dir_not_indexed"])<5:
                        examples["target_dir_not_indexed"].append(f"{sd} + {src} -> {tgt}")

    unresolved=total_cf-resolved
    print(f"\ntotal local cross-file edges: {total_cf}")
    print(f"resolved: {resolved} ({100*resolved/max(total_cf,1):.1f}%)")
    print(f"unresolved: {unresolved} ({100*unresolved/max(total_cf,1):.1f}%)")
    print("\nUnresolved breakdown by cause:")
    for r,c in reasons.most_common():
        pct_all=100*c/max(total_cf,1); pct_unres=100*c/max(unresolved,1)
        print(f"  {r:32s} {c:6d}  ({pct_unres:.1f}% of unresolved, {pct_all:.2f}% of all)")
    print("\nExamples:")
    for r,exs in examples.items():
        print(f"  [{r}]")
        for e in exs: print(f"     {e}")

    out="/content/phase7_out" if os.path.isdir("/content") else "phase7_out"
    os.makedirs(out,exist_ok=True)
    json.dump({"total_cf":total_cf,"resolved":resolved,"unresolved":unresolved,
               "reasons":dict(reasons)}, open(os.path.join(out,"unresolved_breakdown.json"),"w"),indent=2)
    print(f"\nSaved unresolved_breakdown.json to {out}")

def main():
    db=find_db()
    if not db: print("DB not found"); return
    print("DB:",db)
    con=sqlite3.connect(db)
    part_a(con)
    part_b(con)
    con.close()

if __name__=="__main__":
    main()
